# 02 — Run Indexing Pipeline

Embed all GST-registered entity names and build the FAISS index.

**Run this on MAESTRO SageMaker JupyterLab.**  
This is a one-time job (re-run when the GST reference list updates).

Estimated time: ~20 min for 100K entities (depends on API rate limits).

## 1. Install dependencies

In [ ]:
# Uncomment and run if packages are not yet installed
# !pip install openai faiss-cpu boto3 pandas pyarrow numpy python-dotenv

## 2. Set up environment

Make sure your `.env` file is configured with `OPENAI_API_KEY` and `OPENAI_API_BASE`.
On MAESTRO, these point to the LiteLLM proxy.

In [ ]:
import os
import sys

# Ensure the project root is on the Python path
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import (
    S3_BUCKET,
    S3_GST_PREFIX,
    S3_EMBEDDINGS_PREFIX,
    EMBEDDING_MODEL,
    EMBEDDING_DIMENSIONS,
    OPENAI_API_BASE,
)

print(f"S3 bucket:    {S3_BUCKET}")
print(f"GST prefix:   {S3_GST_PREFIX}")
print(f"Output prefix: {S3_EMBEDDINGS_PREFIX}")
print(f"Model:        {EMBEDDING_MODEL}")
print(f"Dimensions:   {EMBEDDING_DIMENSIONS}")
print(f"API base:     {OPENAI_API_BASE[:30]}..." if OPENAI_API_BASE else "API base: NOT SET")

## 3. Quick API connectivity test

Embed a single test string to verify the API is reachable.

In [ ]:
from indexing.embed import embed_names

test_embedding = embed_names(["TEST COMPANY PTE LTD"])
print(f"Shape: {test_embedding.shape}")
print(f"Norm:  {float(test_embedding[0] @ test_embedding[0]):.6f} (should be ~1.0)")
print("API connection OK!")

## 4. Run the full indexing pipeline

This will:
1. Load all entity names from S3 (CSV/parquet files under the GST prefix)
2. Embed all names in batches (100 per request, rate-limited to 50 RPM)
3. Build a FAISS IndexFlatIP index
4. Save the FAISS index + metadata parquet + config JSON to S3

Set `entity_column` below if auto-detection picked the wrong column in notebook 01.

In [ ]:
from indexing.run_indexing import run_indexing

# Set to a column name string to override auto-detection, e.g. "name"
ENTITY_COLUMN = None

run_indexing(entity_column=ENTITY_COLUMN)

## 5. Verify artifacts were saved to S3

In [ ]:
import boto3

s3 = boto3.client("s3")
response = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=S3_EMBEDDINGS_PREFIX)

print(f"Artifacts in s3://{S3_BUCKET}/{S3_EMBEDDINGS_PREFIX}")
for obj in response.get("Contents", []):
    size_mb = obj["Size"] / (1024 * 1024)
    print(f"  {obj['Key']}  ({size_mb:.2f} MB)")

## Next step

Proceed to **`03_test_matching.ipynb`** to test queries against the built index.